In [8]:
# ============================================================
# TUẦN 2 — XÂY DỰNG HỆ THỐNG GỢI Ý SẢN PHẨM
# Hệ thống gợi ý sản phẩm siêu thị Winmart
# ============================================================
# Yêu cầu: pip install pandas scikit-learn openpyxl streamlit

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# ============================================================
# BƯỚC 1: ĐỌC VÀ XỬ LÝ DỮ LIỆU
# Cột có sẵn: Tên sản phẩm, Mô tả ngắn, Thương hiệu, Đơn vị,
#             Giá gốc, Giá sale, Tồn kho, Category, Link
# ============================================================
df = pd.read_excel("sp.xlsx")

# Dùng "Category" làm danh mục
df = df.rename(columns={"Category": "Danh mục"})

# Tính % giảm giá
df["Giảm giá (%)"] = ((df["Giá gốc"] - df["Giá sale"]) / df["Giá gốc"] * 100).round(1)
df["Giảm giá (%)"] = df["Giảm giá (%)"].clip(lower=0).fillna(0)

# Đang giảm giá?
df["Đang giảm giá"] = df["Giảm giá (%)"] > 0

# Còn hàng?
df["Còn hàng"] = df["Tồn kho"].fillna(0) > 0

# Phân khúc theo giá sale
def phan_khuc(gia):
    if pd.isna(gia):
        return "Không rõ"
    if gia < 50000:
        return "Tier 1 — Bình dân"
    elif gia < 200000:
        return "Tier 2 — Trung cấp"
    else:
        return "Tier 3 — Cao cấp"

df["Phân khúc"] = df["Giá sale"].apply(phan_khuc)

# Value Score = mức giảm giá chuẩn hóa (0→1)
gia_max = df["Giá sale"].max()
gia_min = df["Giá sale"].min()
df["Value Score"] = (
    df["Giảm giá (%)"] / 100 * 0.7 +
    (1 - (df["Giá sale"] - gia_min) / (gia_max - gia_min + 1)) * 0.3
).round(4)

print(f"Đọc dữ liệu: {len(df)} sản phẩm | {df['Danh mục'].nunique()} danh mục")
print(f"Đang giảm giá: {df['Đang giảm giá'].sum()} SP")
print(f"Còn hàng: {df['Còn hàng'].sum()} SP")


# ============================================================
# BƯỚC 2: BỘ LỌC THÔNG MINH
# ============================================================

def loc_san_pham(
    tu_khoa=None,
    danh_muc=None,
    gia_min=None,
    gia_max=None,
    thuong_hieu=None,
    chi_giam_gia=False,
    con_hang=True,
    phan_khuc=None,
    sap_xep="value",
    top_n=10,
):
    ket_qua = df.copy()

    if tu_khoa:
        mask = ket_qua["Tên sản phẩm"].str.lower().str.contains(tu_khoa.lower(), na=False)
        ket_qua = ket_qua[mask]

    if danh_muc:
        ket_qua = ket_qua[ket_qua["Danh mục"].str.lower() == danh_muc.lower()]

    if gia_min is not None:
        ket_qua = ket_qua[ket_qua["Giá sale"] >= gia_min]
    if gia_max is not None:
        ket_qua = ket_qua[ket_qua["Giá sale"] <= gia_max]

    if thuong_hieu:
        ket_qua = ket_qua[ket_qua["Thương hiệu"].str.lower() == thuong_hieu.lower()]

    if chi_giam_gia:
        ket_qua = ket_qua[ket_qua["Đang giảm giá"] == True]

    if con_hang:
        ket_qua = ket_qua[ket_qua["Còn hàng"] == True]

    if phan_khuc:
        ket_qua = ket_qua[ket_qua["Phân khúc"].str.lower() == phan_khuc.lower()]

    if sap_xep == "value":
        ket_qua = ket_qua.sort_values("Value Score", ascending=False)
    elif sap_xep == "gia_tang":
        ket_qua = ket_qua.sort_values("Giá sale", ascending=True)
    elif sap_xep == "gia_giam":
        ket_qua = ket_qua.sort_values("Giá sale", ascending=False)
    elif sap_xep == "giam_gia":
        ket_qua = ket_qua.sort_values("Giảm giá (%)", ascending=False)

    cols = ["Tên sản phẩm", "Thương hiệu", "Danh mục",
            "Giá gốc", "Giá sale", "Giảm giá (%)",
            "Phân khúc", "Value Score", "Link"]
    return ket_qua[cols].head(top_n).reset_index(drop=True)


# ============================================================
# BƯỚC 3: MÔ HÌNH GỢI Ý TƯƠNG TỰ (TF-IDF)
# ============================================================

df["features"] = (
    df["Tên sản phẩm"].fillna("") + " " +
    df["Danh mục"].fillna("") + " " +
    df["Thương hiệu"].fillna("") + " " +
    df["Mô tả ngắn"].fillna("")
)

vectorizer = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=1)
tfidf_matrix = vectorizer.fit_transform(df["features"])



def goi_y_tuong_tu(ten_san_pham, top_n=5, chi_cung_danh_muc=True, uu_tien="value"):
    idx_list = df[df["Tên sản phẩm"].str.lower().str.contains(
        ten_san_pham.lower(), na=False
    )].index.tolist()

    if not idx_list:
        print(f"Không tìm thấy: '{ten_san_pham}'")
        return None

    idx = idx_list[0]
    sp_goc = df.loc[idx]
    print(f"\nSản phẩm gốc: {sp_goc['Tên sản phẩm']}")
    print(f"   Giá sale   : {sp_goc['Giá sale']:,}đ")
    print(f"   Giảm giá   : {sp_goc['Giảm giá (%)']:.1f}%")
    print(f"   Danh mục   : {sp_goc['Danh mục']}")
    print(f"   Value Score: {sp_goc['Value Score']:.3f}")

    vec_sp = tfidf_matrix[idx]
    scores = cosine_similarity(vec_sp, tfidf_matrix).flatten()

    sim_df = df.copy()
    sim_df["Độ tương đồng"] = scores
    sim_df = sim_df.drop(index=idx)

    if chi_cung_danh_muc:
        sim_df = sim_df[sim_df["Danh mục"] == sp_goc["Danh mục"]]

    sim_df = sim_df[
        (sim_df["Độ tương đồng"] > 0.05) &
        (sim_df["Còn hàng"] == True)
    ]

    if uu_tien == "value":
        sim_df = sim_df.sort_values(["Value Score", "Độ tương đồng"], ascending=False)
    elif uu_tien == "giam_gia":
        sim_df = sim_df.sort_values(["Giảm giá (%)", "Độ tương đồng"], ascending=False)
    elif uu_tien == "gia_thap":
        sim_df = sim_df.sort_values(["Giá sale", "Độ tương đồng"], ascending=[True, False])

    cols = ["Tên sản phẩm", "Thương hiệu", "Giá sale",
            "Giảm giá (%)", "Value Score", "Độ tương đồng", "Link"]
    return sim_df[cols].head(top_n).reset_index(drop=True)


# ============================================================
# BƯỚC 4: PHÂN TÍCH KHUYẾN MÃI
# ============================================================

def phan_tich_khuyen_mai(danh_muc=None, top_n=10):
    data = df[df["Đang giảm giá"] == True].copy()

    if danh_muc:
        data = data[data["Danh mục"] == danh_muc]

    tb_giam = df.groupby("Danh mục")["Giảm giá (%)"].mean()
    data["TB giảm danh mục (%)"] = data["Danh mục"].map(tb_giam).round(1)
    data["Chênh lệch vs TB (%)"] = (data["Giảm giá (%)"] - data["TB giảm danh mục (%)"]).round(1)

    data["Đánh giá"] = data["Chênh lệch vs TB (%)"].apply(
        lambda x: "Giảm sâu" if x >= 10
        else ("Bình thường" if x >= 0 else "Thấp hơn TB")
    )

    cols = ["Tên sản phẩm", "Thương hiệu", "Danh mục",
            "Giá gốc", "Giá sale", "Giảm giá (%)",
            "TB giảm danh mục (%)", "Chênh lệch vs TB (%)", "Đánh giá"]
    return data[cols].sort_values("Giảm giá (%)", ascending=False).head(top_n).reset_index(drop=True)


# ============================================================
# BƯỚC 5: DEMO
# ============================================================
# ============================================================
# BƯỚC 5: DEMO — THAY ĐỔI Ở ĐÂY
# ============================================================

# ✅ THAY ĐỔI Ở ĐÂY
SP_TIM_GỢI_Ý    = "Máy Vắt Cam Lock&Lock"   # Tên SP muốn tìm tương tự (gõ 1 phần cũng được)
DANH_MỤC_1      = "Bia"                 # Danh mục cho Demo 1 và Demo 4
DANH_MỤC_2      = "Kẹo - Chocolate"          # Danh mục cho Demo 2
GIÁ_TỐI_ĐA      = 100000000                     # Giá tối đa for Demo 1
PHÂN_KHÚC       = "Tier 1 — Bình dân"        # Tier 1 — Bình dân / Tier 2 — Trung cấp / Tier 3 — Cao cấp
SỐ_KẾT_QUẢ     = 5                           # Số SP hiển thị mỗi demo


print("\n" + "="*60)
print(f"DEMO 1 — Lọc: [{DANH_MỤC_1}] giá < {GIÁ_TỐI_ĐA:,}đ, đang giảm giá")
print("="*60)
kq1 = loc_san_pham(danh_muc=DANH_MỤC_1, gia_max=GIÁ_TỐI_ĐA, chi_giam_gia=True, sap_xep="giam_gia", top_n=SỐ_KẾT_QUẢ)
print(kq1[["Tên sản phẩm", "Giá sale", "Giảm giá (%)", "Value Score"]].to_string(index=False))

print("\n" + "="*60)
print(f"DEMO 2 — Lọc: [{DANH_MỤC_2}] phân khúc [{PHÂN_KHÚC}]")
print("="*60)
kq2 = loc_san_pham(danh_muc=DANH_MỤC_2, phan_khuc=PHÂN_KHÚC, sap_xep="value", top_n=SỐ_KẾT_QUẢ)
print(kq2[["Tên sản phẩm", "Giá sale", "Giảm giá (%)", "Value Score"]].to_string(index=False))

print("\n" + "="*60)
print(f"DEMO 3 — Gợi ý SP tương tự: '{SP_TIM_GỢI_Ý}'")
print("="*60)
kq3 = goi_y_tuong_tu(SP_TIM_GỢI_Ý, top_n=SỐ_KẾT_QUẢ, uu_tien="value")
if kq3 is not None:
    print(kq3[["Tên sản phẩm", "Giá sale", "Giảm giá (%)", "Value Score"]].to_string(index=False))

print("\n" + "="*60)
print(f"DEMO 4 — Phân tích khuyến mãi: [{DANH_MỤC_1}]")
print("="*60)
kq4 = phan_tich_khuyen_mai(danh_muc=DANH_MỤC_1, top_n=SỐ_KẾT_QUẢ)
print(kq4[["Tên sản phẩm", "Giá gốc", "Giá sale", "Giảm giá (%)", "TB giảm danh mục (%)", "Đánh giá"]].to_string(index=False))
print("="*60)
kq4 = phan_tich_khuyen_mai(danh_muc=DANH_MỤC_1, top_n=SỐ_KẾT_QUẢ)
print(kq4[["Tên sản phẩm", "Giá gốc", "Giá sale", "Giảm giá (%)", "TB giảm danh mục (%)", "Đánh giá"]].to_string(index=False))



Đọc dữ liệu: 3253 sản phẩm | 75 danh mục
Đang giảm giá: 591 SP
Còn hàng: 3253 SP

DEMO 1 — Lọc: [Bia] giá < 100,000,000đ, đang giảm giá
                            Tên sản phẩm  Giá sale  Giảm giá (%)  Value Score
Bia Sapporo Premium thùng 24 lon x 330ml    418000          13.6       0.3449
          Thùng 12 lon bia Sapporo 500ml    332100          10.7       0.3351
     Thùng 12 lon  Bia  Budweiser 500ml     342600           7.2       0.3093
     Bia 333 Export thùng 24 lon x 330ml    285000           2.1       0.2806
 Thùng 24 lon Budweiser lon Sleek 330ml     443500           1.7       0.2586

DEMO 2 — Lọc: [Kẹo - Chocolate] phân khúc [Tier 1 — Bình dân]
                           Tên sản phẩm  Giá sale  Giảm giá (%)  Value Score
                 Socola Minis M&M'S 35g     24900          29.9       0.5067
 Socola trứng Kinder Joy bé gái hộp 20g     24500          27.7       0.4913
Socola trứng Kinder Joy bé trai hộp 20g     24500          27.7       0.4913
           Socola Sữa M&M